In [ ]:
import sys
sys.path.append("path/to/generated_inference_outputs")
from generate_utils import VLLMGenerationModel
import json
import random
import os 
import pandas as pd
from tqdm import tqdm
os.environ['CUDA_VISIBLE_DEVICES'] = '4,5,6,7'




In [ ]:
META_PROMPT_TEMPLATE = """
Here is a passage written by an English learner whose native language is {native_language}.
Passage: {passage}
Based on this passage, the usage of words and phrases and the writing quality, considering the impact of the learner's native language, infer a vocabulary profile of the learner.
Your should describe the learner's vocabulary capacity and knowledge in details in the profile. 
Do not directly guess the actual vocabulary size of the learner, instead give a detailed description of what the learner may know, and what the learner may not know.
Please only describe things that you are confident about; you only need to give descriptions which are the learner's vocabulary profile; no suggestions.
Return the profile directly.
"""

META_PROMPT_TEMPLATE_no_language = """
Here is a passage written by an English learner.
Passage: {passage}
Based on this passage, the usage of words and phrases and the writing quality, infer a vocabulary profile of the learner.
Your should describe the learner's vocabulary capacity and knowledge in details in the profile. 
Do not directly guess the actual vocabulary size of the learner, instead give a detailed description of what the learner may know, and what the learner may not know.
Please only describe things that you are confident about; you only need to give descriptions which are the learner's vocabulary profile; no suggestions.
Return the profile directly.
"""

def generate_prompt(vllm_model, prompt_list: list[list[dict]], batch_size: int = 20) -> str:
    results = []
    for i in range(0, len(prompt_list), batch_size):
        batch_prompt_list = prompt_list[i:i+batch_size]
        outputs =  vllm_model.generate(batch_prompt_list)
        results.extend(outputs)
    return results

## 1. Sample input texts

In [ ]:

data_path_icnale = "../data/we_all_entries.json"
data_path_teccl = "../data/teccl_all_entries.json"
data_path_fce = "../data/fce_all_entries.json"

#### ICNALE data selection
Based on levels 
ICANEL data does not have C levels;   
Select A2_0 as low level;    
Others random sampling, even distributed by language.    
low-level : others 	$\approx$ 1:1 

In [ ]:
with open(data_path_icnale, "r") as f:
    lines = f.readlines()
icnale_data = [json.loads(line) for line in lines]
icnale_data = pd.DataFrame(icnale_data)


levels = icnale_data['cefr_level'].unique()
print(levels)

def sample_low_level(group, n):

    level_order = list(levels)
    
    sampled_parts = []
    level_df = group[group['cefr_level'] == "A2_0"]
    n = min(n, len(level_df))  # don't oversample
    sampled_parts.append(level_df.sample(n=n, random_state=0))
   
            
    if len(sampled_parts) == 0:
        return None
    return pd.concat(sampled_parts)

def sample_language_group(group, N):
    num_levels = len(levels)
    samples_per_level = N // num_levels
    remainder = N % num_levels

    # Shuffle the levels for fair distribution of remainder
    level_order = list(levels)
    
    sampled_parts = []
    for i, level in enumerate(level_order):
        if level == "A2_0":
            continue
        n = samples_per_level + (1 if i < remainder else 0)  # distribute remainder
        level_df = group[group['cefr_level'] == level]
        n = min(n, len(level_df))  # don't oversample
        sampled_parts.append(level_df.sample(n=n, random_state=0))

    return pd.concat(sampled_parts)

# Apply per language group
icnale_sampled_low_level = icnale_data.groupby('language', group_keys=False).apply(lambda g: sample_low_level(g, 10)).reset_index(drop=True).drop(columns=['id']).rename(columns={'cefr_level': 'level'})
icnale_sampled_df = icnale_data.groupby('language', group_keys=False).apply(lambda g: sample_language_group(g, 10)).reset_index(drop=True).drop(columns=['id']).rename(columns={'cefr_level': 'level'})
icanle_samples = pd.concat([icnale_sampled_low_level, icnale_sampled_df])

In [ ]:
dataset_path = "path/to/L2_learner_corpora/ICNALE/we_all_entries.json"
import json 
import numpy as np
languages = []
text = []
with open(dataset_path) as f:
    for line in f:
        fce_line = json.loads(line)
        languages.append(fce_line["language"])
        text.append(fce_line["text"])
        
print(set(languages))
token_lengths = []
for t in text:
    tokens = t.split(" ")
    len_tokens = len(tokens)
    token_lengths.append(len_tokens)
print(np.mean(token_lengths))

#### Select samples from Teccl data
low level: elementary_school levels, in total 58 samples;    
other levels: 60 samples; 30 each. 

In [ ]:
with open(data_path_teccl, "r") as f:
    lines = f.readlines()
data_teccl = pd.DataFrame([json.loads(line) for line in lines])
teccl_sampled_df_low_level = data_teccl[data_teccl['level'] == "elementary_school"].drop(columns=['id'])
teccl_sampled_df = (
    data_teccl[data_teccl['level'] != "elementary_school"]
    .groupby('level', group_keys=False)
      .apply(lambda x: x.sample(n=min(len(x), 30), random_state=0) )
      .reset_index(drop=True)
).drop(columns=['id'])
teccl_sampleds = pd.concat([teccl_sampled_df_low_level, teccl_sampled_df])



In [ ]:
import pandas as pd
    
dataset_path = "path/to/L2_learner_corpora/TECCL/teccl_all_entries.json"
with open(dataset_path, "r") as f:
    lines = f.readlines()
data_teccl = pd.DataFrame([json.loads(line) for line in lines])
data_teccl['token_count'] = data_teccl['text'].str.split().apply(len)

# Compute average token count
avg_tokens = data_teccl['token_count'].mean()
print(avg_tokens)

#### FCE data  
Select score 1-2 to be low level. 

In [ ]:
with open(data_path_fce, "r") as f:
    lines = f.readlines()
data_fce = pd.DataFrame([json.loads(line) for line in lines])
data_fce['exam_score'] = pd.to_numeric(data_fce['exam_score'], errors='coerce')
# Drop rows where conversion failed (i.e., where exam_score is NaN)
data_fce = data_fce.dropna(subset=['exam_score'])

# Sample 30 entries with exam_score between 1 and 2
low_score_samples = (
    data_fce[(data_fce['exam_score'] >= 1) & (data_fce['exam_score'] <= 2)]
)
# Sample 5 entries per language for exam_score > 2
high_score_samples = (
    data_fce[data_fce['exam_score'] > 2]
    .groupby('language', group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), 5), random_state=0))
    .reset_index(drop=True)
)
low_score_samples['text'] = low_score_samples['incorrect_passage'].apply(lambda x: " ".join(x))
high_score_samples['text'] = high_score_samples['incorrect_passage'].apply(lambda x: " ".join(x))
fce_sampled_low_level = low_score_samples[['text', 'language', 'exam_score']].rename(columns={'exam_score': 'level'})
fce_sampled_df= high_score_samples[['text', 'language', 'exam_score']].rename(columns={'exam_score': 'level'})

fce_samples = pd.concat([fce_sampled_low_level, fce_sampled_df])


In [ ]:
# low_level_samples = pd.concat([icnale_sampled_low_level, teccl_sampled_df_low_level, fce_sampled_low_level])
# mid_level_samples = pd.concat([icnale_sampled_df, teccl_sampled_df, fce_sampled_df])
# low_level_samples.to_json("low_level_samples.json", orient="records", lines=True)
# mid_level_samples.to_json("mid_level_samples.json", orient="records", lines=True)

In [ ]:
dataset_path = "path/to/L2_learner_corpora/fce-released-dataset/fce_all_entries.json"

In [ ]:
import json 
import numpy as np

dataset_path = "path/to/L2_learner_corpora/fce-released-dataset/fce_all_entries.json"
languages = []
score = []
text = []
with open(dataset_path) as f:
    for line in f:
        fce_line = json.loads(line)
        languages.append(fce_line["language"])
        score.append(fce_line["exam_score"])
        text.append(fce_line["incorrect_passage"])
print(set(languages))
print(set(score))
token_lengths = []
for t in text:
    tokens = " ".join(t).split(" ")
    len_tokens = len(tokens)
    token_lengths.append(len_tokens)
print(np.mean(token_lengths))

#### Load samples 

In [ ]:
import json 
import random

with open("low_level_samples.json", "r") as f:
    low_level_samples = [json.loads(line) for line in f]
with open("mid_level_samples.json", "r") as f:
    mid_level_samples = [json.loads(line) for line in f]
with open("coca_text_high_level.json", "r") as f:
    coca_text_high_level = [json.loads(line) for line in f]
    high_level_samples = random.Random(0).sample(coca_text_high_level, 150)

In [ ]:
low_level_samples[0]

In [ ]:
vllm_model = VLLMGenerationModel(model_name_or_path="path/to/local_models/qwen2.5-72b-instruct", gpu_num=4, max_tokens=1000, quantization=True)


In [ ]:
low_level_prompt_list = []
low_level_native_language_list = [sample['language'] for sample in low_level_samples]
low_level_passage_list = [sample['text'] for sample in low_level_samples]
for native_language, passage in zip(low_level_native_language_list, low_level_passage_list):
    content = META_PROMPT_TEMPLATE.format(native_language=native_language, passage=passage)
    prompt = [{"role": "user", "content": content}]
    low_level_prompt_list.append(prompt)

mid_level_prompt_list = []
mid_level_native_language_list = [sample['language'] for sample in mid_level_samples]
mid_level_passage_list = [sample['text'] for sample in mid_level_samples]
for native_language, passage in zip(mid_level_native_language_list, mid_level_passage_list):
    content = META_PROMPT_TEMPLATE.format(native_language=native_language, passage=passage)
    prompt = [{"role": "user", "content": content}]
    mid_level_prompt_list.append(prompt)



high_level_prompt_list = []
for passage in high_level_samples:
    passage = " ".join(passage)
    content = META_PROMPT_TEMPLATE_no_language.format(passage=passage)
    prompt = [{"role": "user", "content": content}]
    high_level_prompt_list.append(prompt)


Generate prompt using meta_prompt list

In [ ]:
low_level_outputs = generate_prompt(vllm_model, low_level_prompt_list, batch_size=20)

In [ ]:
mid_level_outputs = generate_prompt(vllm_model, mid_level_prompt_list, batch_size=20)

In [ ]:
high_level_outputs = generate_prompt(vllm_model, high_level_prompt_list, batch_size=20)

In [ ]:
print(high_level_outputs[0])

In [ ]:
with open("low_level_outputs.json", "w") as f:
    for i in low_level_outputs:
        json.dump(i, f)
        f.write("\n")
with open("mid_level_outputs.json", "w") as f:
    for i in mid_level_outputs:
        json.dump(i, f)
        f.write("\n")

with open("high_level_outputs.json", "w") as f:
    for i in high_level_outputs:
        json.dump(i, f)
        f.write("\n")


In [ ]:
len(low_level_outputs)


In [ ]:
language_list = icnale_sampled_df['language'].tolist()+teccl_sampled_df['language'].tolist()+fce_sampled_df['language'].tolist()
passage_list = icnale_sampled_df['text'].tolist()+teccl_sampled_df['text'].tolist()+fce_sampled_df['text'].tolist()
batch_size = 20
outputs = []
for i in tqdm(range(0, len(language_list), batch_size)):
    batch_language_list = language_list[i:i+batch_size]
    batch_passage_list = passage_list[i:i+batch_size]
    batch_outputs = generate_prompt(vllm_model, batch_language_list, batch_passage_list)
    outputs.extend(batch_outputs)

In [ ]:
print(outputs[5])   

In [ ]:
len(language_list)

In [ ]:
len(outputs)

In [ ]:
data = {
    "language": language_list[:30],
    "prompt": outputs[:30],
}
with open("../data/experiment1_test_meta_prompts.json", "w") as f:
    json.dump(data, f)
